# 17.8 4-bit 如何真的存成較小檔案？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩個小格裝一個byte：低四位與高四位**

本例signed加8：−3→0101，+4→1100，合成11000101。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/int4.svg")))

**先想一想：**7個int4數應佔7bytes還是4bytes？

兩個4bit格可以裝進一個byte，一個放低四位、一個放高四位。奇數長度最後留一個padding格，還原時用原shape裁掉。

**把直覺寫成數學：**packed=low|(high<<4)；unpack分別用&15與>>4，再還原signed偏移。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import pack_int4, unpack_int4

q = torch.tensor([-8, -1, 0, 1, 7, 2, 3], dtype=torch.int8)
packed = pack_int4(q)
restored = unpack_int4(packed, q.shape)
print(q.tolist(), packed.tolist(), "bytes", packed.numel())
assert torch.equal(q, restored) and packed.numel() == 4

**如何讀結果：**這裏真的縮小整數buffer；完整模型文件還含scale、bias與容器header。

**只改一件事：**只把長度改成8，觀察是否仍4bytes。
